# MLD / MCD local avec Mocodo

Ce notebook génère le schéma à partir d'un texte Mocodo en local.

Ordre recommandé:
1. Exécuter la cellule de vérification.
2. Exécuter la cellule qui écrit le fichier source `.mcd`.
3. Exécuter la cellule de génération des sorties.

In [1]:
import subprocess
import sys

print("Python:", sys.version)
result = subprocess.run([sys.executable, "-m", "mocodo", "--version"], capture_output=True, text=True)
print("Code retour:", result.returncode)
print(result.stdout.strip() or result.stderr.strip())

Python: 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
Code retour: 0
4.3.3


In [2]:
from pathlib import Path

mocodo_source = """UTILISATEUR: id utilisateur, email, telephone, password hash, date creation
PANIER: id panier, token session, date creation, date maj
RATTACHER_PANIER, 01 PANIER, 0N UTILISATEUR
COMMANDE: id commande, prenom, nom, email, adresse livraison, departement livraison, pays livraison, statut, montant total, date creation
PASSER_COMMANDE, 01 COMMANDE, 0N UTILISATEUR

LIGNE_PANIER: id ligne panier, quantite, prix unitaire
CONTENIR_LIGNE_PANIER, 11 LIGNE_PANIER, 0N PANIER
PRODUIT: id produit, nom, description, image url, prix base, actif, date creation
CONCERNER_PRODUIT_PANIER, 11 LIGNE_PANIER, 0N PRODUIT
CATEGORIE: id categorie, nom, slug

APPARTENIR, 11 PRODUIT, 0N CATEGORIE
LIGNE_COMMANDE: id ligne commande, quantite, prix unitaire, nom produit snapshot
DETAILLER_COMMANDE, 11 LIGNE_COMMANDE, 1N COMMANDE
CONCERNER_PRODUIT_COMMANDE, 11 LIGNE_COMMANDE, 0N PRODUIT
PAIEMENT: id paiement, methode, reference transaction, montant, statut, date paiement, date creation

REGLER, 11 PAIEMENT, 0N COMMANDE
GROUPE_OPTION: id groupe option, nom
DEFINIR, 11 VALEUR_OPTION, 0N GROUPE_OPTION
VALEUR_OPTION: id valeur option, valeur, surcout
PROPOSER_OPTION, 0N PRODUIT, 0N VALEUR_OPTION

CHOISIR_OPTION_PANIER, 0N LIGNE_PANIER, 0N VALEUR_OPTION
CHOISIR_OPTION_COMMANDE, 0N LIGNE_COMMANDE, 0N VALEUR_OPTION
RECOMMANDER, 0N [source] PRODUIT, 0N [cible] PRODUIT: rang
FAQ_CATEGORIE: id faq categorie, nom, ordre affichage
FAQ_QUESTION: id faq question, question, reponse, ordre affichage, actif

CLASSER_FAQ, 11 FAQ_QUESTION, 0N FAQ_CATEGORIE
MESSAGE_CONTACT: id message, email, sujet, corps, statut, date creation
"""

output_dir = Path("./mocodo_out")
output_dir.mkdir(exist_ok=True)
source_path = output_dir / "ecommerce.mcd"
source_path.write_text(mocodo_source, encoding="utf-8")
print("Source écrite:", source_path.resolve())

Source écrite: /home/saumondeluxe/Desktop/ENSITECH/SIO/ecommerce/bdd/mocodo_out/ecommerce.mcd


In [5]:
import subprocess
import sys
from pathlib import Path

output_dir = Path("./mocodo_out")
source_path = output_dir / "ecommerce.mcd"

cmd = [
    sys.executable,
    "-m",
    "mocodo",
    "--input",
    str(source_path),
    "--output_dir",
    str(output_dir),
]

print("Commande:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print("Code retour:", result.returncode)
if result.stdout.strip():
    print("\nSTDOUT\n", result.stdout)
if result.stderr.strip():
    print("\nSTDERR\n", result.stderr)

print("\nFichiers générés:")
for p in sorted(output_dir.glob("*")):
    print("-", p.name)

print("\nOption PDF (si besoin): pip install cairosvg puis ajoute --svg_to pdf")
print("Astuce réarrangement: python -m mocodo --input mocodo_out/ecommerce.mcd --output_dir mocodo_out -t arrange")

Commande: /home/saumondeluxe/Desktop/ENSITECH/SIO/ecommerce/.venv/bin/python -m mocodo --input mocodo_out/ecommerce.mcd --output_dir mocodo_out
Code retour: 0

Fichiers générés:
- ecommerce.mcd
- ecommerce.svg
- ecommerce_geo.json
- ecommerce_static.svg

Option PDF (si besoin): pip install cairosvg puis ajoute --svg_to pdf
Astuce réarrangement: python -m mocodo --input mocodo_out/ecommerce.mcd --output_dir mocodo_out -t arrange
